# Day 1 exercise: YouTube summarizer

The same steps as Day 1, with a YouTube transcript in place of a website:

1. Get the text (a YouTube transcript instead of `fetch_website_contents`)
2. Write a system prompt and a user prompt
3. Build the `messages` list
4. Call OpenAI and display the result

**One-time setup.** In a terminal, from the `llm_engineering` folder, run:
```
uv add youtube-transcript-api yt-dlp
```

In [ ]:
# imports

import os
import re
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi

In [ ]:
# Load environment variables in a file called .env (same as Day 1)

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-4.1-mini"
ytt = YouTubeTranscriptApi()

## Step 1: Get the transcript
This plays the role of `fetch_website_contents(url)` from Day 1.

In [ ]:
def get_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/|shorts/)([0-9A-Za-z_-]{11})", url)
    if match:
        return match.group(1)
    raise ValueError("Invalid YouTube URL")


def fetch_transcript(url):
    """Returns the video's spoken text. Prefers English, but takes any language."""
    transcripts = list(ytt.list(get_video_id(url)))
    english = [t for t in transcripts if t.language_code.startswith("en")]
    chosen = (english or transcripts)[0]
    return " ".join(snippet.text for snippet in chosen.fetch())

In [ ]:
# Try it out (just like printing `ed` in Day 1)

transcript = fetch_transcript("https://www.youtube.com/watch?v=JCTb3QSrGMQ")
print(transcript[:1000])

## Step 2: The prompts

In [ ]:
system_prompt = """
You analyze transcripts of YouTube videos (podcasts, lectures, advice videos).
List the main claims made in the video. For each claim give:
- the claim in one sentence
- the evidence the speaker gives (study, statistic, expert, personal story, or "none")
- a rating: Strong / Medium / Weak evidence

Then list any products or sponsors the speaker promotes.
Ignore ads and small talk. The transcript may be in any language; always answer in English.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here is the transcript of a YouTube video.
Summarize its main claims and rate the evidence for each one.

"""

## Step 3: The messages (same `messages_for` function as Day 1)

In [ ]:
def messages_for(transcript):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + transcript}
    ]

## Step 4: Call OpenAI

In [ ]:
def summarize(url):
    transcript = fetch_transcript(url)
    response = openai.chat.completions.create(
        model = MODEL,
        messages = messages_for(transcript)
    )
    return response.choices[0].message.content


def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary("https://www.youtube.com/watch?v=JCTb3QSrGMQ")

# Bonus: research a whole topic, not just one video

This uses the same API call again. It summarizes several videos, then asks GPT to compare them:
where they agree, where they disagree, and which claims have weak evidence.

To get videos from other countries, write the topic in another language, for example `"creatina cerebro"`.

In [ ]:
def search_youtube(query, how_many=4):
    """Search YouTube without an API key. Returns a list of video URLs."""
    import yt_dlp
    with yt_dlp.YoutubeDL({"quiet": True, "extract_flat": True}) as ydl:
        results = ydl.extract_info(f"ytsearch{how_many}:{query}", download=False)
    return [f"https://www.youtube.com/watch?v={v['id']}" for v in results["entries"]]


compare_prompt = """
You get summaries of several YouTube videos about one topic.
Write a short report: the bottom line, what the videos agree on, where they disagree,
claims with weak evidence, and who is selling something. Refer to videos as (Video 1), (Video 2) etc.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


def research_topic(topic, how_many=4):
    summaries = []
    for i, url in enumerate(search_youtube(topic, how_many), 1):
        try:
            summaries.append(f"## Video {i}: {url}\n\n{summarize(url)}")
            print(f"Done: {url}")
        except Exception as e:
            print(f"Skipped {url} ({type(e).__name__})")

    if not summaries:
        print("No videos with transcripts found - try another topic")
        return

    all_summaries = "\n\n".join(summaries)
    messages = [
        {"role": "system", "content": compare_prompt},
        {"role": "user", "content": f"Topic: {topic}\n\n{all_summaries}"}
    ]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    display(Markdown(response.choices[0].message.content))

In [ ]:
research_topic("creatine for brain health")